# Alerta Criadouro - API rodando no Google Colab

Global Solution 2026.2 - FIAP - Tecnólogo em IA - turma 1TIAPZ

- Matheus Rebelo da Silva de Jesus - RM 570260
- Felipe Olmos - RM 570946
- Luiz Guilherme - RM 573486

Este notebook liga a nossa API Flask e abre um túnel público da Cloudflare (gratuito, sem cadastro). Assim o ESP32 que roda no Wokwi consegue mandar o JSON para a API pela internet.

## 1. Baixar o projeto do GitHub e o programa do túnel

In [ ]:
!rm -rf alerta-criadouro
!git clone -q https://github.com/matheusrebelo04-maker/alerta-criadouro.git
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
%cd alerta-criadouro/api
!ls

## 2. Ligar a API e o túnel

O token do dispositivo fica numa variável de ambiente, não dentro do código da API.

In [ ]:
import os, subprocess, time, re

os.environ["TOKEN_DISPOSITIVO"] = "ac-quintal-001-demo"
api = subprocess.Popen(["python", "app.py"], stdout=open("servidor.out", "w"), stderr=subprocess.STDOUT)
time.sleep(3)

tunel = subprocess.Popen(["../../cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:5000"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url_publica = None
for linha in tunel.stdout:
    achou = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linha)
    if achou:
        url_publica = achou.group(0)
        break
print("API no ar em:", url_publica)
print("Endereco para colocar no ESP32:", url_publica + "/leituras")

## 3. Testar os 3 cenários pela internet

In [ ]:
time.sleep(5)
!TOKEN_DISPOSITIVO=ac-quintal-001-demo python testar_api.py {url_publica}

## 4. Ver o que foi gravado

Rode de novo depois que o Wokwi mandar as leituras.

In [ ]:
import pandas as pd
pd.read_csv("leituras.csv").tail(15)

In [ ]:
!tail -n 20 seguranca.log